# Exploratory Data Analysis

This notebook documents the data-loading and leakage-removal steps used for the readmission model. It shows the target binarization, leaky discharge filtering, missing-value handling strategy, and the demographic features that later feed the bias audit.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
repo_root = Path.cwd()
for candidate in [repo_root, repo_root.parent, repo_root / '..']:
    candidate = candidate.resolve()
    if (candidate / 'data' / 'diabetic_data.csv').exists():
        repo_root = candidate
        break

data_path = repo_root / 'data' / 'diabetic_data.csv'
df = pd.read_csv(data_path)
print(f"Raw dataset shape: {df.shape}")
df.head()


In [ ]:
df['target'] = (df['readmitted'] == '<30').astype(int)
print(df['target'].value_counts(normalize=True))

fig, ax = plt.subplots(figsize=(6, 4), dpi=120)
sns.countplot(x='target', data=df, ax=ax, palette=['#7f8c8d', '#e67e22'])
ax.set_title('Binarized Target Distribution')
ax.set_xlabel('Target (0 = >30/NO, 1 = <30)')
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()


In [ ]:
leakage_ids = [11, 13, 14, 19, 20, 21]
leakage_df = df[df['discharge_disposition_id'].isin(leakage_ids)]
df_clean = df[~df['discharge_disposition_id'].isin(leakage_ids)].copy()
print(f'Leaky rows removed: {len(leakage_df)}')
print(f'Clean dataset shape: {df_clean.shape}')

missing_cols = df_clean.replace('?', np.nan).isna().mean().sort_values(ascending=False)
print('\nTop missingness rates:')
print((missing_cols[missing_cols > 0] * 100).head(10).round(2))

for column in ['race', 'gender', 'age']:
    print(f'\n{column} distribution:')
    print(df_clean[column].value_counts(dropna=False).head(8))
